In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib, gc, os

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'
EMBED_PATH = f'{BASE_PATH}/embeddings'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Mounted at /content/drive
Device: cuda


In [ ]:
train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_features.csv')

paths = {
    'contextual': {
        'train_emb': f'{EMBED_PATH}/train_embeddings_final.npy', 'test_emb': f'{EMBED_PATH}/test_embeddings_final.npy',
        'train_idx': f'{EMBED_PATH}/train_embeddings_index.csv', 'test_idx': f'{EMBED_PATH}/test_embeddings_index.csv',
    },
    'temporal': {
        'train_emb': f'{EMBED_PATH}/train_temporal_embeddings.npy', 'test_emb': f'{EMBED_PATH}/test_temporal_embeddings.npy',
        'train_idx': f'{EMBED_PATH}/train_temporal_embeddings_index.csv', 'test_idx': f'{EMBED_PATH}/test_temporal_embeddings_index.csv',
    },
    'behavioral': {
        'train_emb': f'{EMBED_PATH}/train_behavioral_embeddings.npy', 'test_emb': f'{EMBED_PATH}/test_behavioral_embeddings.npy',
        'train_idx': f'{EMBED_PATH}/train_behavioral_embeddings_index.csv', 'test_idx': f'{EMBED_PATH}/test_behavioral_embeddings_index.csv',
    },
}

branch_data = {}
for branch, p in paths.items():
    branch_data[branch] = {
        'train_emb': np.load(p['train_emb']), 'test_emb': np.load(p['test_emb']),
        'train_ids': pd.read_csv(p['train_idx'])['email_id'].values,
        'test_ids': pd.read_csv(p['test_idx'])['email_id'].values,
    }

print(train_df.shape, test_df.shape)

/tmp/ipykernel_986/2470410455.py:1: DtypeWarning: Columns (21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')


(202583, 43) (50646, 43)


In [ ]:
def dedupe_ids(ids, emb):
    seen = set()
    keep_mask = np.zeros(len(ids), dtype=bool)
    for i, eid in enumerate(ids):
        if eid not in seen:
            keep_mask[i] = True
            seen.add(eid)
    return ids[keep_mask], emb[keep_mask]

train_df = train_df.drop_duplicates(subset='email_id', keep='first').copy()
test_df = test_df.drop_duplicates(subset='email_id', keep='first').copy()

for branch in branch_data:
    for split in ['train', 'test']:
        ids = branch_data[branch][f'{split}_ids']
        emb = branch_data[branch][f'{split}_emb']
        new_ids, new_emb = dedupe_ids(ids, emb)
        branch_data[branch][f'{split}_ids'] = new_ids
        branch_data[branch][f'{split}_emb'] = new_emb

print("train_df:", len(train_df), "| test_df:", len(test_df))

train_df: 202582 | test_df: 50645


In [ ]:
assert train_df['email_id'].is_unique and test_df['email_id'].is_unique
for branch in branch_data:
    assert pd.Series(branch_data[branch]['train_ids']).is_unique
    assert pd.Series(branch_data[branch]['test_ids']).is_unique
print("All IDs unique across raw tables and all branches. Confirmed clean.")

All IDs unique across raw tables and all branches. Confirmed clean.


In [ ]:
enron_subj = pd.read_csv(f'{BASE_PATH}/enron_emails_cleaned_temporal.csv', usecols=['message_id', 'subject']).rename(columns={'message_id': 'email_id'})

nazario_raw = pd.read_csv(f'{BASE_PATH}/nazario5_cleaned_temporal.csv', usecols=['sender_address', 'utc_datetime', 'subject'])
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_subj = nazario_raw[['email_id', 'subject']]

subjects_all = pd.concat([enron_subj, nazario_subj], ignore_index=True).drop_duplicates(subset='email_id')

train_df = train_df.merge(subjects_all, on='email_id', how='left')
test_df = test_df.merge(subjects_all, on='email_id', how='left')

train_df['text'] = "[SUBJECT] " + train_df['subject'].fillna('') + " [BODY] " + train_df['body_text'].fillna('')
test_df['text'] = "[SUBJECT] " + test_df['subject'].fillna('') + " [BODY] " + test_df['body_text'].fillna('')

del enron_subj, nazario_raw, nazario_subj, subjects_all
gc.collect()
print(train_df.shape, test_df.shape)

(202582, 45) (50645, 45)


In [ ]:
SEQ_FEATURES = ['log_interarrival','hour_sin','hour_cos','day_sin','day_cos','is_weekend','is_burst_anomaly','temporal_anomaly']
MAX_SEQ_LEN = 10

def prep_temporal_features(df):
    df = df.copy()
    df['log_interarrival'] = np.log1p(df['interarrival_hours'].fillna(0).clip(lower=0))
    df['hour_sin'] = np.sin(2*np.pi*df['hour_of_day_utc']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour_of_day_utc']/24)
    df['day_sin'] = np.sin(2*np.pi*df['day_of_week']/7)
    df['day_cos'] = np.cos(2*np.pi*df['day_of_week']/7)
    df['is_weekend'] = df['is_weekend'].astype(float)
    df['is_burst_anomaly'] = df['is_burst_anomaly'].astype(float)
    df['temporal_anomaly'] = df['temporal_anomaly'].fillna(False).astype(float)
    df['sender'] = df['sender'].fillna('__unknown_sender__')
    return df

def build_sequences(df, seq_features=SEQ_FEATURES, max_len=MAX_SEQ_LEN):
    df2 = df.sort_values(['sender','datetime']).reset_index(drop=True)
    feat_matrix = df2[seq_features].values.astype(np.float32)
    n = len(df2)
    sequences = np.zeros((n, max_len, len(seq_features)), dtype=np.float32)
    lengths = np.zeros(n, dtype=np.int64)
    for _, idx_group in df2.groupby('sender').indices.items():
        idx_group = np.sort(idx_group)
        for pos, row_idx in enumerate(idx_group):
            start = max(0, pos - max_len + 1)
            window_idx = idx_group[start:pos+1]
            L = len(window_idx)
            sequences[row_idx, :L, :] = feat_matrix[window_idx]
            lengths[row_idx] = L
    lengths = np.maximum(lengths, 1)
    return sequences, lengths, df2['email_id'].values

def reindex_to(target_ids, source_ids, *arrays):
    pos = {eid: i for i, eid in enumerate(source_ids)}
    order = [pos[eid] for eid in target_ids]
    return tuple(arr[order] for arr in arrays)

train_df = prep_temporal_features(train_df)
test_df = prep_temporal_features(test_df)

print("Building train sequences...")
seq_train_raw, lens_train_raw, ids_train_seq = build_sequences(train_df)
print("Building test sequences...")
seq_test_raw, lens_test_raw, ids_test_seq = build_sequences(test_df)

train_seq, train_seq_lens = reindex_to(train_df['email_id'].values, ids_train_seq, seq_train_raw, lens_train_raw)
test_seq, test_seq_lens = reindex_to(test_df['email_id'].values, ids_test_seq, seq_test_raw, lens_test_raw)

print(train_seq.shape, test_seq.shape)
del seq_train_raw, seq_test_raw
gc.collect()

Building train sequences...
Building test sequences...
(202582, 10, 8) (50645, 10, 8)


0

In [ ]:
BEHAVIORAL_COLS = [
    'domain_age_days', 'domain_reputation_score', 'tld_risk_weight',
    'has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass',
    'ip_is_residential_proxy', 'ip_is_vpn_or_anon', 'ip_is_datacenter', 'ip_reputation_score',
    'sender_email_count', 'sender_active_days', 'sender_avg_daily_volume',
    'unique_recipient_count', 'is_repeat_correspondent', 'recipient_reuse_ratio', 'sender_prior_unique_recipients',
    'has_url', 'is_ip_based', 'subdomain_count', 'path_length', 'query_param_count', 'url_length', 'uses_https',
]

def clean_behavioral(df):
    df = df.copy()
    domain_cols = ['domain_age_days', 'domain_reputation_score', 'tld_risk_weight']
    df['domain_unknown'] = df['domain_age_days'].isna().astype(float)
    for col in domain_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in ['has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass']:
        df[col] = df[col].fillna(False).astype(float)
    url_cols = ['is_ip_based','subdomain_count','path_length','query_param_count','url_length','uses_https']
    for col in url_cols:
        df[col] = df[col].fillna(0).astype(float)
    for col in ['ip_is_residential_proxy','ip_is_vpn_or_anon','ip_is_datacenter','is_repeat_correspondent']:
        df[col] = df[col].astype(float)
    return df

FINAL_BEHAVIORAL_COLS = BEHAVIORAL_COLS + ['domain_unknown']

train_df = clean_behavioral(train_df)
test_df = clean_behavioral(test_df)

scaler = joblib.load(f'{BASE_PATH}/behavioral_scaler.pkl')
train_behav = scaler.transform(train_df[FINAL_BEHAVIORAL_COLS].values.astype(np.float32))
test_behav = scaler.transform(test_df[FINAL_BEHAVIORAL_COLS].values.astype(np.float32))

print(train_behav.shape, test_behav.shape)

/tmp/ipykernel_986/1882758692.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_986/1882758692.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_986/1882758692.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
 

(202582, 26) (50645, 26)


In [ ]:
assert train_seq.shape[0] == len(train_df) == train_behav.shape[0]
assert test_seq.shape[0] == len(test_df) == test_behav.shape[0]
assert not np.isnan(train_seq).any() and not np.isnan(train_behav).any()
assert not np.isnan(test_seq).any() and not np.isnan(test_behav).any()

print("Train rows:", len(train_df), "| seq:", train_seq.shape, "| behav:", train_behav.shape)
print("Test rows:", len(test_df), "| seq:", test_seq.shape, "| behav:", test_behav.shape)
print("All aligned. No NaNs. Ready for model assembly.")

Train rows: 202582 | seq: (202582, 10, 8) | behav: (202582, 26)
Test rows: 50645 | seq: (50645, 10, 8) | behav: (50645, 26)
All aligned. No NaNs. Ready for model assembly.


training

In [ ]:
from transformers import AutoTokenizer, DistilBertModel

FINAL_MODEL_DIR = f'{BASE_PATH}/baseline_distilbert_finetuned'
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)

MAX_LENGTH = 256

class TemporalLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return h_n[-1]

class BehavioralFFN(nn.Module):
    def __init__(self, input_dim=26, hidden_dims=(64, 32)):
        super().__init__()
        self.hidden1 = nn.Sequential(nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(), nn.Dropout(0.2))
        self.hidden2 = nn.Sequential(nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(0.2))
    def forward(self, x):
        return self.hidden2(self.hidden1(x))

class FusionModel(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(FINAL_MODEL_DIR)  # warm-started, old classifier head correctly ignored
        self.lstm = TemporalLSTM()
        self.ffn = BehavioralFFN()
        self.fusion_head = nn.Sequential(
            nn.Linear(768 + 64 + 32, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
        self.register_buffer('class_weights', class_weights)

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask, seq, seq_lengths, behav, labels=None):
        contextual_out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        contextual_emb = self.mean_pool(contextual_out, attention_mask)

        temporal_emb = self.lstm(seq, seq_lengths)
        behavioral_emb = self.ffn(behav)

        fused = torch.cat([contextual_emb, temporal_emb, behavioral_emb], dim=1)
        logits = self.fusion_head(fused)

        output = {'logits': logits}
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
            output['loss'] = loss_fct(logits, labels)
        return output

# class weights from the label distribution
label_counts = train_df['label'].value_counts().sort_index()
total = label_counts.sum()
class_weights = torch.tensor([total/(2*c) for c in label_counts], dtype=torch.float32)

# assemble model, warm-start each branch from its previously saved weights
model = FusionModel(class_weights)

lstm_state = torch.load(f'{BASE_PATH}/temporal_lstm_model.pt')
lstm_filtered = {k: v for k, v in lstm_state.items() if k.startswith('lstm.')}
model.lstm.load_state_dict(lstm_filtered)

ffn_state = torch.load(f'{BASE_PATH}/behavioral_ffn_model.pt')
ffn_filtered = {k: v for k, v in ffn_state.items() if k.startswith('hidden')}
model.ffn.load_state_dict(ffn_filtered)

model = model.to(device)
print("Model assembled, warm-started from all 3 saved branches.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: /content/drive/My Drive/phishing project datasets/processed/baseline_distilbert_finetuned
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model assembled, warm-started from all 3 saved branches.


In [ ]:
class FusionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, seqs, seq_lens, behav, labels):
        self.texts = texts
        self.seqs = seqs
        self.seq_lens = seq_lens
        self.behav = behav
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'seq': self.seqs[idx],
            'seq_len': self.seq_lens[idx],
            'behav': self.behav[idx],
            'label': self.labels[idx],
        }

def collate_fn(batch):
    texts = [b['text'] for b in batch]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
    seq = torch.tensor(np.stack([b['seq'] for b in batch]), dtype=torch.float32)
    seq_len = torch.tensor([b['seq_len'] for b in batch], dtype=torch.long)
    behav = torch.tensor(np.stack([b['behav'] for b in batch]), dtype=torch.float32)
    labels = torch.tensor([b['label'] for b in batch], dtype=torch.long)
    return {
        'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
        'seq': seq, 'seq_lengths': seq_len, 'behav': behav, 'labels': labels
    }

train_dataset = FusionDataset(train_df['text'].tolist(), train_seq, train_seq_lens, train_behav, train_df['label'].values)
test_dataset = FusionDataset(test_df['text'].tolist(), test_seq, test_seq_lens, test_behav, test_df['label'].values)

print(len(train_dataset), len(test_dataset))

202582 50645


In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score

class FusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        labels = inputs.get('labels')
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.get('loss')
            logits = outputs['logits']
        return (loss, logits, labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'precision': precision, 'recall': recall, 'f1': f1,
        'roc_auc': roc_auc_score(labels, probs),
        'pr_auc': average_precision_score(labels, probs)
    }

In [ ]:
class FusionCollator:
    def __call__(self, features):
        return collate_fn(features)

data_collator = FusionCollator()

In [ ]:
MODEL_CHECKPOINT_DIR = f'{BASE_PATH}/fusion_checkpoints'
os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,   # effective batch size 32, same as Task 2
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc",
    report_to="none",
    dataloader_num_workers=0,        # same fix as Task 2 -- avoids the torchvision worker crash
    remove_unused_columns=False,     # required: our Dataset returns a dict Trainer doesn't recognize by default
    label_names=["labels"],
)

trainer = FusionTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
import glob
existing_checkpoints = glob.glob(f'{MODEL_CHECKPOINT_DIR}/checkpoint-*')
resume = bool(existing_checkpoints)
print("Resuming from checkpoint:", resume)

trainer.train(resume_from_checkpoint=resume if resume else None)

Resuming from checkpoint: False


Step,Training Loss,Validation Loss,Precision,Recall,F1,Roc Auc,Pr Auc
2000,0.014715,0.028204,0.986742,0.875630,0.927872,0.999350,0.977412
4000,0.010031,0.020373,0.989305,0.932773,0.960208,0.999526,0.986044
6000,0.005598,0.014280,0.975779,0.947899,0.961637,0.999642,0.988086
6331,0.008415,0.013939,0.970890,0.952941,0.961832,0.999641,0.987769


TrainOutput(global_step=6331, training_loss=0.016931171007507354, metrics={'train_runtime': 2960.4602, 'train_samples_per_second': 68.429, 'train_steps_per_second': 2.139, 'total_flos': 0.0, 'train_loss': 0.016931171007507354, 'epoch': 1.0})

In [ ]:
FUSION_MODEL_DIR = f'{BASE_PATH}/fusion_model_final'
os.makedirs(FUSION_MODEL_DIR, exist_ok=True)

torch.save(model.state_dict(), f'{FUSION_MODEL_DIR}/fusion_model.pt')
tokenizer.save_pretrained(FUSION_MODEL_DIR)

print(f"Saved fusion model to: {FUSION_MODEL_DIR}")

Saved fusion model to: /content/drive/My Drive/phishing project datasets/processed/fusion_model_final


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(test_dataset)
logits = predictions.predictions
labels = predictions.label_ids
preds = logits.argmax(axis=1)

print(classification_report(labels, preds, target_names=['benign', 'phishing'], digits=4))
print("\nConfusion matrix:")
print(confusion_matrix(labels, preds))

              precision    recall  f1-score   support

      benign     0.9994    0.9997    0.9996     50050
    phishing     0.9758    0.9479    0.9616       595

    accuracy                         0.9991     50645
   macro avg     0.9876    0.9738    0.9806     50645
weighted avg     0.9991    0.9991    0.9991     50645


Confusion matrix:
[[50036    14]
 [   31   564]]
